# Lab 08 

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, ConfusionMatrixDisplay

In [ ]:
url = 'https://raw.githubusercontent.com/rfordatascience/tidytuesday/master/data/2020/2020-02-11/hotels.csv'
df = pd.read_csv(url)

df.head()

In [ ]:
df['total_nights'] = df['stays_in_weekend_nights'] + df['stays_in_week_nights']

X = df[['lead_time', 'adr', 'previous_cancellations', 'total_nights']]
y = df['is_canceled']

data = X.copy()
data['is_canceled'] = y
data = data.replace([np.inf, -np.inf], np.nan).dropna()

X = data[['lead_time', 'adr', 'previous_cancellations', 'total_nights']]
y = data['is_canceled'].astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=42
)

print('Training data:', X_train.shape)
print('Testing data:', X_test.shape)

In [ ]:
log_model = make_pipeline(
    StandardScaler(),
    LogisticRegression(C=1.0, penalty='l2', solver='liblinear', max_iter=2000)
)

mlp_model = make_pipeline(
    StandardScaler(),
    MLPClassifier(hidden_layer_sizes=(8,), activation='relu',
                  solver='adam', alpha=0.001, max_iter=500, random_state=42)
)

log_model.fit(X_train, y_train)
mlp_model.fit(X_train, y_train)

log_pred = log_model.predict(X_test)
mlp_pred = mlp_model.predict(X_test)

In [ ]:
results = []

for name, pred in [('Logistic Regression', log_pred), ('MLP', mlp_pred)]:
    results.append({
        'Model': name,
        'Accuracy': round(accuracy_score(y_test, pred), 3),
        'Precision': round(precision_score(y_test, pred, zero_division=0), 3),
        'Recall': round(recall_score(y_test, pred, zero_division=0), 3),
        'F1': round(f1_score(y_test, pred, zero_division=0), 3)
    })

results_df = pd.DataFrame(results)
results_df

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

ConfusionMatrixDisplay.from_predictions(
    y_test, log_pred, display_labels=['Not canceled', 'Canceled'],
    ax=axes[0], colorbar=False
)
axes[0].set_title('Logistic Regression')

ConfusionMatrixDisplay.from_predictions(
    y_test, mlp_pred, display_labels=['Not canceled', 'Canceled'],
    ax=axes[1], colorbar=False
)
axes[1].set_title('MLP')

plt.tight_layout()
plt.show()

In [ ]:
plt.plot(mlp_model[-1].loss_curve_)
plt.xlabel('Iteration')
plt.ylabel('Loss')
plt.title('MLP Training Loss')
plt.grid(True)
plt.show()

### Business insights

1. Recall shows how many actual cancellations the model can identify.
2. Low precision means the hotel may contact guests who would not cancel.
3. The hotel should compare recall, precision and F1 before choosing a model, depending on whether missed cancellations or unnecessary follow-ups are more costly.